# EgyptGPT Autoresearch

Autonomous AI research loop using Claude Code on your Claude subscription.
Based on Karpathy's [autoresearch](https://github.com/karpathy/autoresearch) pattern.

**Requirements:** Colab GPU runtime, Claude Pro/Max subscription.

## 1. Clone repo and install dependencies

In [ ]:
%cd /content
!rm -rf EgyptGPT
!git clone --recurse-submodules https://github.com/JLansey/EgyptGPT.git
%cd /content/EgyptGPT
!bash setup.sh

## 2. Mount Google Drive and prepare data

In [ ]:
import os, shutil
from google.colab import drive
drive.mount('/content/drive')

%cd /content/EgyptGPT

DATA_DIR = 'data/egypt_char'
DRIVE_DATA = '/content/drive/MyDrive/EgyptGPT_autoresearch/data'
DATA_FILES = ['train.bin', 'val.bin', 'meta.pkl']

# Try loading cached data from Drive
cached = all(os.path.exists(os.path.join(DRIVE_DATA, f)) for f in DATA_FILES)
if cached:
    print('Loading cached data from Google Drive...')
    for f in DATA_FILES:
        shutil.copy(os.path.join(DRIVE_DATA, f), os.path.join(DATA_DIR, f))
else:
    print('No cache. Running prepare.py...')
    # Must run as module import so hiero_transformer submodule is on the path
    !cd /content/EgyptGPT && python -c "from data.egypt_char import prepare"
    # Cache to Drive
    os.makedirs(DRIVE_DATA, exist_ok=True)
    for f in DATA_FILES:
        shutil.copy(os.path.join(DATA_DIR, f), os.path.join(DRIVE_DATA, f))
    print(f'Cached to {DRIVE_DATA}')

# Verify — fail hard if anything is missing
for f in DATA_FILES:
    path = os.path.join(DATA_DIR, f)
    assert os.path.exists(path), f'MISSING: {path}'
    print(f'{path}: {os.path.getsize(path):,} bytes')
print('Data ready.')

## 3. Install Claude Code

In [ ]:
!curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash -
!sudo apt-get install -y nodejs tmux
!npm install -g @anthropic-ai/claude-code
!claude --version

## 4. Configure git

In [ ]:
%cd /content/EgyptGPT
!git config user.email "autoresearch@colab"
!git config user.name "autoresearch"

## 5. Log in to Claude (one-time, uses your subscription)

Open the **Colab terminal** (bottom-left terminal icon) and run:

```bash
cd /content/EgyptGPT && claude
```

It will show an OAuth URL. **Copy the URL, paste it in your browser, and log in.**
Once authenticated, your credentials are stored for this session.

Type `/exit` to quit Claude Code, then come back here and run the next cell.

## 6. Launch autoresearch

Each experiment takes ~5 minutes. Expect ~12/hour, ~100 overnight.

**To watch:** `tmux attach -t autoresearch` in the Colab terminal

**To stop:** `tmux kill-session -t autoresearch`

In [ ]:
import subprocess

subprocess.run(['tmux', 'kill-session', '-t', 'autoresearch'], capture_output=True)

prompt = """Read program.md in this directory. This is an autoresearch setup.

1. Create branch autoresearch/run1 from master
2. Verify data/egypt_char/train.bin and val.bin exist
3. Initialize results.tsv with the header row
4. Begin the experiment loop as described in program.md

After each experiment, copy results.tsv to /content/drive/MyDrive/EgyptGPT_autoresearch/results.tsv so the log survives session crashes.

Start with the baseline run, then iterate autonomously. Never stop."""

cmd = f'cd /content/EgyptGPT && claude -p {repr(prompt)} --dangerously-skip-permissions'
subprocess.run(['tmux', 'new-session', '-d', '-s', 'autoresearch', cmd])

print('Autoresearch running in tmux.')
print('Watch:  tmux attach -t autoresearch  (in Colab terminal)')
print('Check:  !cat /content/EgyptGPT/results.tsv')
print('Stop:   !tmux kill-session -t autoresearch')

## 7. Monitor

In [ ]:
!cat /content/EgyptGPT/results.tsv 2>/dev/null || echo 'No results yet.'

In [ ]:
!cd /content/EgyptGPT && git log --oneline -20

In [ ]:
!tmux has-session -t autoresearch 2>/dev/null && echo 'Running' || echo 'Stopped'